# Queries

Verification SQL queries against `database/books.db`.

Every query below is written as an SQL string, executed via Python's
built-in `sqlite3` driver, and returned through `pandas.read_sql_query()`
so it renders as a nice table in this notebook.

Each query cell is preceded by a markdown cell that explains the goal
and the SQL feature being demonstrated.

In [ ]:
import os
import sqlite3
import pandas as pd

PROJECT_ROOT = os.path.dirname(os.path.abspath("queries.ipynb"))
DB_PATH = os.path.join(PROJECT_ROOT, "database", "books.db")

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        f"Database not found at {DB_PATH}. "
        "Run `python database.py` first to build it."
    )

# Use a single connection for the whole notebook so cells stay independent.
conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")

# Quick schema peek so the rest of the notebook makes sense.
schema = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn,
)
schema

## Q1. Row counts

Confirm the database is healthy: 4 categories, 88 books, 88 unique titles.
If `books != unique_titles` the database still has duplicates.

In [ ]:
sql_q1 = """
SELECT
    (SELECT COUNT(*)          FROM categories) AS categories,
    (SELECT COUNT(*)          FROM books)      AS books,
    (SELECT COUNT(DISTINCT title) FROM books)   AS unique_titles
"""

pd.read_sql_query(sql_q1, conn)

## Q2. Books with their category name (JOIN)

Demonstrates `JOIN` between `books` and `categories`. This proves every
book is tagged with a real category (Travel, Mystery, Historical Fiction,
Classics) rather than the old hard-coded `"All Books"`.

In [ ]:
sql_q2 = """
SELECT b.book_id,
       b.title,
       c.category_name,
       b.price_inr,
       b.rating,
       b.in_stock
FROM   books      AS b
JOIN   categories AS c ON b.category_id = c.category_id
ORDER  BY c.category_name, b.title
"""

pd.read_sql_query(sql_q2, conn)

## Q3. Books per category (GROUP BY)

Aggregate row counts and average rating per category. Demonstrates
`GROUP BY` together with `COUNT` and `AVG`.

In [ ]:
sql_q3 = """
SELECT c.category_name,
       COUNT(*)                AS book_count,
       ROUND(AVG(b.rating), 2) AS avg_rating
FROM   books      AS b
JOIN   categories AS c ON b.category_id = c.category_id
GROUP  BY c.category_name
ORDER  BY book_count DESC
"""

pd.read_sql_query(sql_q3, conn)

## Q4. Categories priced above the global mean (HAVING)

Demonstrates `HAVING`: filter groups by an aggregate condition. The
subquery returns the global average price, and we keep only the
categories that beat it.

In [ ]:
sql_q4 = """
SELECT c.category_name,
       ROUND(AVG(b.price_inr), 2) AS avg_price_inr
FROM   books      AS b
JOIN   categories AS c ON b.category_id = c.category_id
GROUP  BY c.category_name
HAVING AVG(b.price_inr) > (
    SELECT AVG(price_inr) FROM books
)
ORDER  BY avg_price_inr DESC
"""

pd.read_sql_query(sql_q4, conn)

## Q5. Average INR price per category

Min, average, and max INR price per category, sorted by the average.
Demonstrates multiple aggregate functions in one query.

In [ ]:
sql_q5 = """
SELECT c.category_name,
       ROUND(AVG(b.price_inr), 2) AS avg_price_inr,
       ROUND(MIN(b.price_inr), 2) AS min_price_inr,
       ROUND(MAX(b.price_inr), 2) AS max_price_inr
FROM   books      AS b
JOIN   categories AS c ON b.category_id = c.category_id
GROUP  BY c.category_name
ORDER  BY avg_price_inr DESC
"""

pd.read_sql_query(sql_q5, conn)

## Q6. Top 5 most expensive in-stock books

Demonstrates `WHERE`, `ORDER BY`, and `LIMIT` together. Filters to books
that are actually in stock and returns the five most expensive.

In [ ]:
sql_q6 = """
SELECT b.title,
       c.category_name,
       b.price_inr,
       b.rating
FROM   books      AS b
JOIN   categories AS c ON b.category_id = c.category_id
WHERE  b.in_stock = 1
ORDER  BY b.price_inr DESC
LIMIT  5
"""

pd.read_sql_query(sql_q6, conn)

## Summary

Across these six queries we verified:

- **Schema**: two tables (`categories`, `books`) with a foreign key.
- **No duplicates**: `books` rows equal `DISTINCT title` rows.
- **Real categories**: every book joins to Travel / Mystery / Historical Fiction / Classics.
- **Aggregate behaviour**: counts, averages, and `HAVING` filters all return sensible numbers.
- **Operational read path**: filtering + ordering + limiting produces the expected top-5.